In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/bureau_balance_sample.csv
/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/HomeCredit_columns_description.csv
/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/instalments_payments_sample.csv
/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/POS_CASH_balance_sample.csv
/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/bureau_sample.csv
/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/previous_application_sample.csv
/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/application_train_sample.csv
/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/credit_card_balance_sample.csv


# Notebook Purpose

Đây là notebook dùng để lọc dữ liệu (e.g. loại bỏ các cột không cần thiết,...), encoding các biến trong bảng phụ để mà có thể tổng hợp (aggregate) về cấp độ SK_ID_CURR, sau đó merge với bảng chính (application_train) theo SK_ID_CURR. 

**Note**: Đây là experiment notebook dùng để thử nghiệm cách merge hiệu quả giữa các tập dataset nên không có các datapipeline để tự động fit train, transform test

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import matplotlib.ticker as mticker
import math
import itertools
from typing import Literal
from matplotlib.ticker import FuncFormatter
from matplotlib.axes import Axes
from matplotlib.figure import Figure
import seaborn.objects as so
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [3]:
application_train = pd.read_csv('/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/application_train_sample.csv')
bureau = pd.read_csv('/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/bureau_sample.csv')
bureau_balance = pd.read_csv('/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/bureau_balance_sample.csv')
credit_card_balance = pd.read_csv('/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/credit_card_balance_sample.csv')
installments_payments = pd.read_csv('/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/instalments_payments_sample.csv')
previous_application = pd.read_csv('/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/previous_application_sample.csv')
POS_CASH_balance = pd.read_csv('/kaggle/input/datasets/thuyduc/home-credit-default-risk-sample/POS_CASH_balance_sample.csv')

In [4]:
from sklearn.model_selection import train_test_split

X = application_train.drop(columns=['TARGET'])
y = application_train['TARGET']

# chia tập dữ liệu cho application_train
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, stratify=y_train, test_size=0.2, random_state=42)

SK_ID_CURR_train = X_train['SK_ID_CURR'].values
SK_ID_CURR_val = X_val['SK_ID_CURR'].values
SK_ID_CURR_test = X_test['SK_ID_CURR'].values

# chia tập dữ liệu cho bureau
bureau_train = bureau[bureau['SK_ID_CURR'].isin(SK_ID_CURR_train)]
bureau_val = bureau[bureau['SK_ID_CURR'].isin(SK_ID_CURR_val)]
bureau_test = bureau[bureau['SK_ID_CURR'].isin(SK_ID_CURR_test)]

SK_ID_BUREAU_train = bureau_train['SK_ID_BUREAU'].values
SK_ID_BUREAU_val = bureau_val['SK_ID_BUREAU'].values
SK_ID_BUREAU_test = bureau_test['SK_ID_BUREAU'].values

# chia tập dữ liệu cho previous_application
previous_application_train = previous_application[previous_application['SK_ID_CURR'].isin(SK_ID_CURR_train)]
previous_application_val = previous_application[previous_application['SK_ID_CURR'].isin(SK_ID_CURR_val)]
previous_application_test = previous_application[previous_application['SK_ID_CURR'].isin(SK_ID_CURR_test)]

SK_ID_PREV_train = previous_application_train['SK_ID_PREV'].values
SK_ID_PREV_val = previous_application_val['SK_ID_PREV'].values
SK_ID_PREV_test = previous_application_test['SK_ID_PREV'].values

# chia tập dữ liệu cho bureau_balance
bureau_balance_train = bureau_balance[bureau_balance['SK_ID_BUREAU'].isin(SK_ID_BUREAU_train)]
bureau_balance_val = bureau_balance[bureau_balance['SK_ID_BUREAU'].isin(SK_ID_BUREAU_val)]
bureau_balance_test = bureau_balance[bureau_balance['SK_ID_BUREAU'].isin(SK_ID_BUREAU_test)]

# chia tập dữ liệu cho POS_CASH_balance
filter_pos_cash_train = (POS_CASH_balance['SK_ID_CURR'].isin(SK_ID_CURR_train)) & (POS_CASH_balance['SK_ID_PREV'].isin(SK_ID_PREV_train))
filter_pos_cash_val = (POS_CASH_balance['SK_ID_CURR'].isin(SK_ID_CURR_val)) & (POS_CASH_balance['SK_ID_PREV'].isin(SK_ID_PREV_val))
filter_pos_cash_test = (POS_CASH_balance['SK_ID_CURR'].isin(SK_ID_CURR_test)) & (POS_CASH_balance['SK_ID_PREV'].isin(SK_ID_PREV_test))

POS_CASH_balance_train = POS_CASH_balance[filter_pos_cash_train]
POS_CASH_balance_val = POS_CASH_balance[filter_pos_cash_val]
POS_CASH_balance_test = POS_CASH_balance[filter_pos_cash_test]

# chia tập dữ liệu cho instalments_payments
filter_instalments_train = (installments_payments['SK_ID_CURR'].isin(SK_ID_CURR_train)) & (installments_payments['SK_ID_PREV'].isin(SK_ID_PREV_train))
filter_instalments_val = (installments_payments['SK_ID_CURR'].isin(SK_ID_CURR_val)) & (installments_payments['SK_ID_PREV'].isin(SK_ID_PREV_val))
filter_instalments_test = (installments_payments['SK_ID_CURR'].isin(SK_ID_CURR_test)) & (installments_payments['SK_ID_PREV'].isin(SK_ID_PREV_test))

instalments_payments_train = installments_payments[filter_instalments_train]
instalments_payments_val = installments_payments[filter_instalments_val]
instalments_payments_test = installments_payments[filter_instalments_test]

# chia tập dữ liệu cho credit_card_balance
filter_credit_train = (credit_card_balance['SK_ID_CURR'].isin(SK_ID_CURR_train)) & (credit_card_balance['SK_ID_PREV'].isin(SK_ID_PREV_train))
filter_credit_val = (credit_card_balance['SK_ID_CURR'].isin(SK_ID_CURR_val)) & (credit_card_balance['SK_ID_PREV'].isin(SK_ID_PREV_val))
filter_credit_test = (credit_card_balance['SK_ID_CURR'].isin(SK_ID_CURR_test)) & (credit_card_balance['SK_ID_PREV'].isin(SK_ID_PREV_test))

credit_card_balance_train = credit_card_balance[filter_credit_train]
credit_card_balance_val = credit_card_balance[filter_credit_val]
credit_card_balance_test = credit_card_balance[filter_credit_test]

In [5]:
# Define Custom Transformer
class RemoveFeature(BaseEstimator, TransformerMixin):
    def __init__(self, columns_removed:list[str]):
        """
        columns_removed: columns which will be removed from dataset
        """
        self.columns_removed = columns_removed

    def fit(self, X:pd.DataFrame, y=None):
        self.is_fitted_ = True
        return self

    def transform(self, X:pd.DataFrame):
        all_cols = X.columns
        columns_selected = set(all_cols) - set(self.columns_removed)
        self.columns_selected = list(columns_selected)
        X_new = X[self.columns_selected]
        return X_new

    def get_feature_names_out(self, input_features=None): # must have this method
        # must have 'input_features' parameter
        return self.columns_selected

# Tiền xử lý cho application_train


In [6]:
# loại bỏ một vài cột không cần thiết dựa trên EDA
application_train_aggregate = application_train.isnull().sum().astype(float).to_frame()
application_train_aggregate.iloc[:, 0] = application_train_aggregate.iloc[:, 0] / len(application_train)
application_train_aggregate = application_train_aggregate.rename(columns={0: "missing_ratio"})
data = application_train_aggregate[application_train_aggregate['missing_ratio']!=0]
data = data.reset_index()

# tìm và loại bỏ các cột có quá nhiều missing values
filter_index = data['missing_ratio'].sort_values(ascending=True).index[:9]
columns_name_keep = data.iloc[filter_index, :]['index'].values
all_columns = data['index'].values
columns_name_remove = set(all_columns) - set(columns_name_keep)
columns_name_remove = list(columns_name_remove)

other_columns_remove = ['FLAG_MOBIL', 'FLAG_CONT_MOBILE', 'FLAG_EMAIL', 'REG_REGION_NOT_LIVE_REGION',\
                        'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY',\
                       'FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_7',\
                       'FLAG_DOCUMENT_8', 'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_11', 'FLAG_DOCUMENT_12',\
                       'FLAG_DOCUMENT_13', 'FLAG_DOCUMENT_14', 'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_16', 'FLAG_DOCUMENT_17',\
                       'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20', 'FLAG_DOCUMENT_21', 'WEEKDAY_APPR_PROCESS_START']
columns_name_remove.extend(other_columns_remove)

In [7]:
feature_removed_application_train = RemoveFeature(columns_removed = columns_name_remove)

# Create Pipeline
application_train_clean_pipeline = Pipeline([
    ('remove_features', feature_removed_application_train),
])

application_train_transformed = application_train_clean_pipeline.fit_transform(application_train)

# Tiền xử lý bureau

In [8]:
bureau_aggregate = bureau.isnull().sum().astype(float).to_frame()
bureau_aggregate.iloc[:, 0] = bureau_aggregate.iloc[:, 0] / len(bureau)
bureau_aggregate = bureau_aggregate.rename(columns={0: "missing_ratio"})
data = bureau_aggregate[bureau_aggregate['missing_ratio']!=0]
data = data.reset_index()

# tìm và loại bỏ các cột có quá nhiều missing values
filter_index = data['missing_ratio'].sort_values(ascending=True).index[:2]
columns_name_keep = data.iloc[filter_index, :]['index'].values
all_columns = data['index'].values
columns_name_remove = set(all_columns) - set(columns_name_keep)
columns_name_remove = list(columns_name_remove)

other_columns_remove = ['CREDIT_CURRENCY', 'CREDIT_DAY_OVERDUE', 'CNT_CREDIT_PROLONG']
columns_name_remove.extend(other_columns_remove)

In [9]:
feature_removed_bureau = RemoveFeature(columns_removed = columns_name_remove)

# Create Pipeline
bureau_clean_pipeline = Pipeline([
    ('remove_features', feature_removed_bureau),
])

bureau_transform = bureau_clean_pipeline.fit_transform(bureau)

# Tiền xử lý credit_card_balance

In [10]:
credit_card_balance_aggregate = credit_card_balance.isnull().sum().astype(float).to_frame()
credit_card_balance_aggregate.iloc[:, 0] = credit_card_balance_aggregate.iloc[:, 0] / len(credit_card_balance)
credit_card_balance_aggregate = credit_card_balance_aggregate.rename(columns={0: "missing_ratio"})
data = credit_card_balance_aggregate[credit_card_balance_aggregate['missing_ratio']!=0]
data = data.reset_index()

# tìm và loại bỏ các cột có quá nhiều missing values
filter_index = data['missing_ratio'].sort_values(ascending=True).index[:2]
columns_name_keep = data.iloc[filter_index, :]['index'].values
all_columns = data['index'].values
columns_name_remove = set(all_columns) - set(columns_name_keep)
columns_name_remove = list(columns_name_remove)

other_columns_remove = ['SK_DPD', 'SK_DPD_DEF']
columns_name_remove.extend(other_columns_remove)

In [11]:
feature_removed_credit_card_balance = RemoveFeature(columns_removed = columns_name_remove)

# Create Pipeline
credit_card_clean_pipeline = Pipeline([
    ('remove_features', feature_removed_credit_card_balance),
])

credit_card_balance_transformed = credit_card_clean_pipeline.fit_transform(credit_card_balance)

# Tiền xử lý previous_application

In [12]:
previous_application_aggregate = previous_application.isnull().sum().astype(float).to_frame()
previous_application_aggregate.iloc[:, 0] = previous_application_aggregate.iloc[:, 0] / len(previous_application)
previous_application_aggregate = previous_application_aggregate.rename(columns={0: "missing_ratio"})
data = previous_application_aggregate[previous_application_aggregate['missing_ratio']!=0]
data = data.reset_index()

# tìm và loại bỏ các cột có quá nhiều missing values
filter_index = data['missing_ratio'].sort_values(ascending=True).index[:1]
columns_name_keep = data.iloc[filter_index, :]['index'].values
all_columns = data['index'].values
columns_name_remove = set(all_columns) - set(columns_name_keep)
columns_name_remove = list(columns_name_remove)

other_columns_remove = ['NFLAG_LAST_APPL_IN_DAY', 'FLAG_LAST_APPL_PER_CONTRACT', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START']
columns_name_remove.extend(other_columns_remove)

In [13]:
feature_removed_previous_application = RemoveFeature(columns_removed = columns_name_remove)

# Create Pipeline
previous_application_clean_pipeline = Pipeline([
    ('remove_features', feature_removed_previous_application),
])

previous_application_transform = previous_application_clean_pipeline.fit_transform(previous_application)

In [14]:
bureau_balance_transform = bureau_balance.copy()
installments_payments_transform = installments_payments.copy()

In [15]:
columns_name_remove = ['SK_DPD', 'SK_DPD_DEF']

feature_removed_POS_CASH_balance = RemoveFeature(columns_removed = columns_name_remove)

# Create Pipeline
pos_cas_balance_clean_pipeline = Pipeline([
    ('remove_features', feature_removed_POS_CASH_balance),
])

POS_CASH_balance_transform = pos_cas_balance_clean_pipeline.fit_transform(POS_CASH_balance)

# Merge bảng chính và bảng phụ

BB: bureau_balance

B: bureau

AT: application_train

CC: credit card

PA: previous application

PC: pos cash

IP: installments payments

In [16]:
# Thêm tiền tố vào các bảng bureau trước
bureau_transform_columns = list(bureau_transform.columns)
alter_columns_name_bureau = []

for column in bureau_transform_columns:
    if column not in ['SK_ID_CURR', 'SK_ID_BUREAU']:
        alter_columns_name_bureau.append(f'B_{column}')
    else:
        alter_columns_name_bureau.append(column)

bureau_transform.columns = alter_columns_name_bureau

# Thêm tiền tố vào bảng bureau_balance
bureau_balance_transform_columns = list(bureau_balance_transform.columns)
alter_columns_name_bureau_balance = []

for column in bureau_balance_transform_columns:
    if column not in ['SK_ID_BUREAU']:
        alter_columns_name_bureau_balance.append(f'BB_{column}')
    else:
        alter_columns_name_bureau_balance.append(column)

bureau_balance_transform.columns = alter_columns_name_bureau_balance

# Thêm tiền tố vào bảng application_train
application_train_transform_columns = list(application_train_transformed.columns)
alter_columns_name_application_train = []

for column in application_train_transform_columns:
    if column not in ['SK_ID_CURR']:
        alter_columns_name_application_train.append(f'AT_{column}')
    else:
        alter_columns_name_application_train.append(column)

application_train_transformed.columns = alter_columns_name_application_train

# Thêm tiền tố vào bảng credit_card_balance
credit_card_balance_transform_columns = list(credit_card_balance_transformed.columns)
alter_columns_name_credit_card_balance = []

for column in credit_card_balance_transform_columns:
    if column not in ['SK_ID_CURR', 'SK_ID_PREV']:
        alter_columns_name_credit_card_balance.append(f'CC_{column}')
    else:
        alter_columns_name_credit_card_balance.append(column)

credit_card_balance_transformed.columns = alter_columns_name_credit_card_balance

# Thêm tiền tố vào bảng POS_CASH_balance
POS_CASH_balance_transform_columns = list(POS_CASH_balance_transform.columns)
alter_columns_name_POS_CASH_balance = []

for column in POS_CASH_balance_transform_columns:
    if column not in ['SK_ID_CURR', 'SK_ID_PREV']:
        alter_columns_name_POS_CASH_balance.append(f'PC_{column}')
    else:
        alter_columns_name_POS_CASH_balance.append(column)

POS_CASH_balance_transform.columns = alter_columns_name_POS_CASH_balance

# Thêm tiền tố vào bảng installments_payments
installments_payments_transform_columns = list(installments_payments_transform.columns)
alter_columns_name_installments_payments = []

for column in installments_payments_transform_columns:
    if column not in ['SK_ID_CURR', 'SK_ID_PREV']:
        alter_columns_name_installments_payments.append(f'IP_{column}')
    else:
        alter_columns_name_installments_payments.append(column)

installments_payments_transform.columns = alter_columns_name_installments_payments

# Thêm tiền tố vào bảng previous_application
previous_application_transform_columns = list(previous_application_transform.columns)
alter_columns_name_previous_application = []

for column in previous_application_transform_columns:
    if column not in ['SK_ID_CURR', 'SK_ID_PREV']:
        alter_columns_name_previous_application.append(f'PA_{column}')
    else:
        alter_columns_name_previous_application.append(column)

previous_application_transform.columns = alter_columns_name_previous_application

In [17]:
# thực hiện aggregation cho bureau_balance

bb_encoded = pd.get_dummies(bureau_balance_transform, columns=['BB_STATUS'], prefix='BB_STATUS')
bb_encoded
status_cols = [col for col in bb_encoded.columns if col.startswith('BB_STATUS_')]
agg_dict = {
    'BB_MONTHS_BALANCE': ['min', 'max', 'mean']
}
for col in status_cols:
    agg_dict[col] = ['sum', 'mean']

bb_agg = bb_encoded.groupby('SK_ID_BUREAU').agg(agg_dict)

bb_agg.columns = [f'{col[0]}_{col[1].upper()}' for col in bb_agg.columns]
bb_agg.reset_index(inplace=True)

bb_agg.head()

,SK_ID_BUREAU,BB_MONTHS_BALANCE_MIN,BB_MONTHS_BALANCE_MAX,BB_MONTHS_BALANCE_MEAN,BB_STATUS_0_SUM,BB_STATUS_0_MEAN,BB_STATUS_1_SUM,BB_STATUS_1_MEAN,BB_STATUS_2_SUM,BB_STATUS_2_MEAN,BB_STATUS_3_SUM,BB_STATUS_3_MEAN,BB_STATUS_4_SUM,BB_STATUS_4_MEAN,BB_STATUS_5_SUM,BB_STATUS_5_MEAN,BB_STATUS_C_SUM,BB_STATUS_C_MEAN,BB_STATUS_X_SUM,BB_STATUS_X_MEAN
0,5008804,-15,0,-7.5,1,0.062500,1,0.062500,0,0.000000,0,0.000000,0,0.000000,0,0.000000,13,0.812500,1,0.062500
1,5008805,-14,0,-7.0,1,0.066667,1,0.066667,0,0.000000,0,0.000000,0,0.000000,0,0.000000,12,0.800000,1,0.066667
2,5008815,-5,0,-2.5,6,1.000000,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.000000
3,5008818,-86,0,-43.0,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.000000,0,0.000000,85,0.977011,2,0.022989
4,5008827,-37,0,-18.5,2,0.052632,8,0.210526,2,0.052632,1,0.026316,1,0.026316,5,0.131579,19,0.500000,0,0.000000


In [18]:
bb_agg.shape

(105107, 20)

In [19]:
# merge bb_agg với bureau_transform qua SK_ID_BUREAU
# 1. Merge dữ liệu
merge_dataset_with_bureau = bureau_transform.merge(bb_agg, on='SK_ID_BUREAU', how='left')
merge_dataset_with_bureau.drop(columns=['SK_ID_BUREAU'], inplace=True)

# 2. One-Hot Encoding cùng lúc cho cả 2 cột (Tránh lỗi ghi đè)
merge_dataset_with_bureau_encoded = pd.get_dummies(
    merge_dataset_with_bureau,
    columns=['B_CREDIT_ACTIVE', 'B_CREDIT_TYPE'],
    prefix=['B_CREDIT_ACTIVE', 'B_CREDIT_TYPE'],
)

# 3. Phân loại các cột One-Hot và cột định lượng
cat_cols = [
    col
    for col in merge_dataset_with_bureau_encoded.columns
    if col.startswith(('B_CREDIT_ACTIVE_', 'B_CREDIT_TYPE_'))
]
num_cols = list(
    merge_dataset_with_bureau_encoded.select_dtypes(include=['number']).columns
)
num_cols = [col for col in num_cols if col not in cat_cols and col != 'SK_ID_CURR']

# 4. Xây dựng agg_dict tối ưu hơn
agg_dict = {}
for col in cat_cols:
  agg_dict[col] = ['sum', 'mean']

for col in num_cols:
  agg_dict[col] = ['min', 'max', 'mean', 'sum']

# 5. Groupby theo SK_ID_CURR
merge_dataset_with_bureau_agg = merge_dataset_with_bureau_encoded.groupby('SK_ID_CURR').agg(agg_dict)

# 6. Làm phẳng tên cột
merge_dataset_with_bureau_agg.columns = [
    f'{col[0]}_{col[1].upper()}' for col in merge_dataset_with_bureau_agg.columns
]
merge_dataset_with_bureau_agg.reset_index(inplace=True)
merge_dataset_with_bureau_agg.head()

# merge_dataset_with_bureau_agg sẽ được merge vào trong application_train_transformed sau

,SK_ID_CURR,B_CREDIT_ACTIVE_Active_SUM,B_CREDIT_ACTIVE_Active_MEAN,B_CREDIT_ACTIVE_Bad debt_SUM,B_CREDIT_ACTIVE_Bad debt_MEAN,B_CREDIT_ACTIVE_Closed_SUM,B_CREDIT_ACTIVE_Closed_MEAN,B_CREDIT_ACTIVE_Sold_SUM,B_CREDIT_ACTIVE_Sold_MEAN,B_CREDIT_TYPE_Another type of loan_SUM,...,BB_STATUS_C_MEAN_MEAN,BB_STATUS_C_MEAN_SUM,BB_STATUS_X_SUM_MIN,BB_STATUS_X_SUM_MAX,BB_STATUS_X_SUM_MEAN,BB_STATUS_X_SUM_SUM,BB_STATUS_X_MEAN_MIN,BB_STATUS_X_MEAN_MAX,BB_STATUS_X_MEAN_MEAN,BB_STATUS_X_MEAN_SUM
0,100019,2,1.000000,0,0.0,0,0.000000,0,0.0,0,...,0.0,0.0,1.0,1.0,1.0,2.0,0.058824,0.058824,0.058824,0.117647
1,100023,4,0.307692,0,0.0,9,0.692308,0,0.0,0,...,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.000000
2,100027,1,0.333333,0,0.0,2,0.666667,0,0.0,0,...,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.000000
3,100031,3,0.428571,0,0.0,4,0.571429,0,0.0,0,...,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.000000
4,100036,1,1.000000,0,0.0,0,0.000000,0,0.0,0,...,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.000000


In [20]:
# tổng hợp pos_cash_balance theo SK_ID_PREV
POS_CASH_balance_encoded = pd.get_dummies(
    POS_CASH_balance_transform,
    columns=['PC_NAME_CONTRACT_STATUS'],
    prefix=['PC_NAME_CONTRACT_STATUS'],
)
POS_CASH_balance_encoded.drop(columns=['SK_ID_CURR'], inplace=True)

# 3. Phân loại các cột One-Hot và cột định lượng
cat_cols = [
    col
    for col in POS_CASH_balance_encoded.columns
    if col.startswith(('PC_NAME_CONTRACT_STATUS_'))
]
num_cols = list(
    POS_CASH_balance_encoded.select_dtypes(include=['number']).columns
)
num_cols = [col for col in num_cols if col not in cat_cols and col != 'SK_ID_PREV']

# 4. Xây dựng agg_dict tối ưu hơn
agg_dict = {}
for col in cat_cols:
  agg_dict[col] = ['sum', 'mean']

for col in num_cols:
  agg_dict[col] = ['min', 'max', 'mean', 'sum']

# 5. Groupby theo SK_ID_PREV
POS_CASH_balance_agg = POS_CASH_balance_encoded.groupby('SK_ID_PREV').agg(agg_dict)

# 6. Làm phẳng tên cột
POS_CASH_balance_agg.columns = [
    f'{col[0]}_{col[1].upper()}' for col in POS_CASH_balance_agg.columns
]
POS_CASH_balance_agg.reset_index(inplace=True)
POS_CASH_balance_agg.head()

,SK_ID_PREV,PC_NAME_CONTRACT_STATUS_Active_SUM,PC_NAME_CONTRACT_STATUS_Active_MEAN,PC_NAME_CONTRACT_STATUS_Amortized debt_SUM,PC_NAME_CONTRACT_STATUS_Amortized debt_MEAN,PC_NAME_CONTRACT_STATUS_Approved_SUM,PC_NAME_CONTRACT_STATUS_Approved_MEAN,PC_NAME_CONTRACT_STATUS_Canceled_SUM,PC_NAME_CONTRACT_STATUS_Canceled_MEAN,PC_NAME_CONTRACT_STATUS_Completed_SUM,...,PC_MONTHS_BALANCE_MEAN,PC_MONTHS_BALANCE_SUM,PC_CNT_INSTALMENT_FUTURE_MIN,PC_CNT_INSTALMENT_FUTURE_MAX,PC_CNT_INSTALMENT_FUTURE_MEAN,PC_CNT_INSTALMENT_FUTURE_SUM,PC_CNT_INSTALMENT_MIN,PC_CNT_INSTALMENT_MAX,PC_CNT_INSTALMENT_MEAN,PC_CNT_INSTALMENT_SUM
0,1000005,10,0.909091,0,0.0,0,0.0,0,0.0,1,...,-51.0,-561,0.0,10.0,5.000000,55.0,10.0,10.0,10.0,110.0
1,1000045,10,0.909091,0,0.0,0,0.0,0,0.0,1,...,-90.0,-990,0.0,9.0,4.090909,45.0,9.0,9.0,9.0,99.0
2,1000049,4,0.800000,0,0.0,0,0.0,0,0.0,1,...,-10.0,-50,0.0,12.0,8.400000,42.0,4.0,12.0,10.4,52.0
3,1000050,10,1.000000,0,0.0,0,0.0,0,0.0,0,...,-5.5,-55,3.0,12.0,7.500000,75.0,12.0,12.0,12.0,120.0
4,1000055,10,0.909091,0,0.0,0,0.0,0,0.0,1,...,-34.0,-374,0.0,10.0,4.363636,48.0,10.0,10.0,10.0,110.0


In [21]:
# tổng hợp installments_payments theo SK_ID_PREV
installments_payments_encoded = installments_payments_transform.copy()

installments_payments_encoded.drop(columns=['SK_ID_CURR'], inplace=True)

num_cols = list(
    installments_payments_encoded.select_dtypes(include=['number']).columns
)
num_cols = [col for col in num_cols if col != 'SK_ID_PREV']

# 4. Xây dựng agg_dict tối ưu hơn
agg_dict = {}
for col in num_cols:
  agg_dict[col] = ['min', 'max', 'mean', 'sum']

# 5. Groupby theo SK_ID_PREV
installments_payments_agg = installments_payments_encoded.groupby('SK_ID_PREV').agg(agg_dict)

# 6. Làm phẳng tên cột
installments_payments_agg.columns = [
    f'{col[0]}_{col[1].upper()}' for col in installments_payments_agg.columns
]
installments_payments_agg.reset_index(inplace=True)
installments_payments_agg.head()

,SK_ID_PREV,IP_NUM_INSTALMENT_VERSION_MIN,IP_NUM_INSTALMENT_VERSION_MAX,IP_NUM_INSTALMENT_VERSION_MEAN,IP_NUM_INSTALMENT_VERSION_SUM,IP_NUM_INSTALMENT_NUMBER_MIN,IP_NUM_INSTALMENT_NUMBER_MAX,IP_NUM_INSTALMENT_NUMBER_MEAN,IP_NUM_INSTALMENT_NUMBER_SUM,IP_DAYS_INSTALMENT_MIN,...,IP_DAYS_ENTRY_PAYMENT_MEAN,IP_DAYS_ENTRY_PAYMENT_SUM,IP_AMT_INSTALMENT_MIN,IP_AMT_INSTALMENT_MAX,IP_AMT_INSTALMENT_MEAN,IP_AMT_INSTALMENT_SUM,IP_AMT_PAYMENT_MIN,IP_AMT_PAYMENT_MAX,IP_AMT_PAYMENT_MEAN,IP_AMT_PAYMENT_SUM
0,1000005,1.0,1.0,1.000000,11.0,1,10,5.818182,64,-1688.0,...,-1551.909091,-17071.0,14599.260,14713.605,14703.210000,161735.310,2.790,14713.605,13365.609545,147021.705
1,1000018,0.0,0.0,0.000000,0.0,1,17,9.000000,153,-161.0,...,-78.705882,-1338.0,40.635,6938.370,2073.994412,35257.905,40.635,6938.370,2073.994412,35257.905
2,1000045,1.0,2.0,1.111111,10.0,1,9,5.000000,45,-2874.0,...,-2785.000000,-25065.0,3712.140,7317.360,4112.720000,37014.480,3712.140,7317.360,4112.720000,37014.480
3,1000049,1.0,6.0,3.500000,21.0,1,4,2.166667,13,-315.0,...,-290.000000,-1740.0,1692.360,14592.690,8539.020000,51234.120,1692.360,18000.000,14539.020000,87234.120
4,1000050,1.0,1.0,1.000000,10.0,1,10,5.500000,55,-298.0,...,-184.800000,-1848.0,13833.450,13833.450,13833.450000,138334.500,13833.450,13833.450,13833.450000,138334.500


In [22]:
# tổng hợp credit_card_balance theo SK_ID_PREV
credit_card_balance_encoded = pd.get_dummies(
    credit_card_balance_transformed,
    columns=['CC_NAME_CONTRACT_STATUS'],
    prefix=['CC_NAME_CONTRACT_STATUS'],
)
credit_card_balance_encoded.drop(columns=['SK_ID_CURR'], inplace=True)

# 3. Phân loại các cột One-Hot và cột định lượng
cat_cols = [
    col
    for col in credit_card_balance_encoded.columns
    if col.startswith(('CC_NAME_CONTRACT_STATUS_'))
]
num_cols = list(
    credit_card_balance_encoded.select_dtypes(include=['number']).columns
)
num_cols = [col for col in num_cols if col not in cat_cols and col != 'SK_ID_PREV']

# 4. Xây dựng agg_dict tối ưu hơn
agg_dict = {}
for col in cat_cols:
  agg_dict[col] = ['sum', 'mean']

for col in num_cols:
  agg_dict[col] = ['min', 'max', 'mean', 'sum']

# 5. Groupby theo SK_ID_PREV
credit_card_balance_agg = credit_card_balance_encoded.groupby('SK_ID_PREV').agg(agg_dict)

# 6. Làm phẳng tên cột
credit_card_balance_agg.columns = [
    f'{col[0]}_{col[1].upper()}' for col in credit_card_balance_agg.columns
]
credit_card_balance_agg.reset_index(inplace=True)
credit_card_balance_agg.head()

,SK_ID_PREV,CC_NAME_CONTRACT_STATUS_Active_SUM,CC_NAME_CONTRACT_STATUS_Active_MEAN,CC_NAME_CONTRACT_STATUS_Completed_SUM,CC_NAME_CONTRACT_STATUS_Completed_MEAN,CC_NAME_CONTRACT_STATUS_Demand_SUM,CC_NAME_CONTRACT_STATUS_Demand_MEAN,CC_NAME_CONTRACT_STATUS_Refused_SUM,CC_NAME_CONTRACT_STATUS_Refused_MEAN,CC_NAME_CONTRACT_STATUS_Sent proposal_SUM,...,CC_AMT_PAYMENT_TOTAL_CURRENT_MEAN,CC_AMT_PAYMENT_TOTAL_CURRENT_SUM,CC_AMT_BALANCE_MIN,CC_AMT_BALANCE_MAX,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_SUM,CC_AMT_RECEIVABLE_PRINCIPAL_MIN,CC_AMT_RECEIVABLE_PRINCIPAL_MAX,CC_AMT_RECEIVABLE_PRINCIPAL_MEAN,CC_AMT_RECEIVABLE_PRINCIPAL_SUM
0,1000018,5,1.000000,0,0.000000,0,0.0,0,0.0,0,...,5541.750000,27708.750,38879.145,136695.420,74946.285000,374731.425,37542.645,132903.000,72298.197000,361490.985
1,1000281,6,1.000000,0,0.000000,0,0.0,0,0.0,0,...,8081.077500,48486.465,0.000,48399.030,12850.185000,77101.110,0.000,45000.000,12147.015000,72882.090
2,1000286,3,0.272727,8,0.727273,0,0.0,0,0.0,0,...,8583.263182,94415.895,0.000,93407.940,16673.040000,183403.440,0.000,88425.000,16077.272727,176850.000
3,1000466,83,1.000000,0,0.000000,0,0.0,0,0.0,0,...,1915.481928,158985.000,0.000,94086.000,20835.692349,1729362.465,0.000,90000.000,19998.456506,1659871.890
4,1000547,10,1.000000,0,0.000000,0,0.0,0,0.0,0,...,21750.120000,217501.200,10027.980,215362.125,91066.837500,910668.375,10027.980,213506.235,89560.597500,895605.975


In [23]:
# merge 3 bảng balance với previous_application, sau đó aggregate bảng được merge theo SK_ID_CURR
merge_dataset_with_previous_application = previous_application_transform.merge(credit_card_balance_agg, on='SK_ID_PREV', how='left')
merge_dataset_with_previous_application = merge_dataset_with_previous_application.merge(POS_CASH_balance_agg, on='SK_ID_PREV', how='left')
merge_dataset_with_previous_application = merge_dataset_with_previous_application.merge(installments_payments_agg, on='SK_ID_PREV', how='left')
merge_dataset_with_previous_application.drop(columns=['SK_ID_PREV'], inplace=True)

category_cols = list(merge_dataset_with_previous_application.select_dtypes(include=['object']).columns)

merge_dataset_with_previous_application_encoded = pd.get_dummies(
    merge_dataset_with_previous_application,
    columns=[col for col in category_cols],
    prefix=[col for col in category_cols],
)

# 3. Phân loại các cột One-Hot và cột định lượng
cat_cols = [
    col
    for col in merge_dataset_with_previous_application_encoded.columns
    if col.startswith(tuple([f"{column}_" for column in category_cols]))
]
num_cols = list(
    merge_dataset_with_previous_application_encoded.select_dtypes(include=['number']).columns
)
num_cols = [col for col in num_cols if col not in cat_cols and col != 'SK_ID_CURR']

# 4. Xây dựng agg_dict tối ưu hơn
agg_dict = {}
for col in cat_cols:
  agg_dict[col] = ['sum', 'mean']

for col in num_cols:
  agg_dict[col] = ['min', 'max', 'mean', 'sum']

# 5. Groupby theo SK_ID_CURR
merge_dataset_with_previous_application_agg = merge_dataset_with_previous_application_encoded.groupby('SK_ID_CURR').agg(agg_dict)

# 6. Làm phẳng tên cột
merge_dataset_with_previous_application_agg.columns = [
    f'{col[0]}_{col[1].upper()}' for col in merge_dataset_with_previous_application_agg.columns
]
merge_dataset_with_previous_application_agg.reset_index(inplace=True)
merge_dataset_with_previous_application_agg.head()

# merge_dataset_with_previous_application_agg sẽ được merge vào trong application_train_transformed sau

/tmp/ipykernel_58/1992337120.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merge_dataset_with_previous_application_agg.reset_index(inplace=True)


,SK_ID_CURR,PA_NAME_PRODUCT_TYPE_XNA_SUM,PA_NAME_PRODUCT_TYPE_XNA_MEAN,PA_NAME_PRODUCT_TYPE_walk-in_SUM,PA_NAME_PRODUCT_TYPE_walk-in_MEAN,PA_NAME_PRODUCT_TYPE_x-sell_SUM,PA_NAME_PRODUCT_TYPE_x-sell_MEAN,PA_NAME_SELLER_INDUSTRY_Auto technology_SUM,PA_NAME_SELLER_INDUSTRY_Auto technology_MEAN,PA_NAME_SELLER_INDUSTRY_Clothing_SUM,...,IP_AMT_PAYMENT_MAX_MEAN,IP_AMT_PAYMENT_MAX_SUM,IP_AMT_PAYMENT_MEAN_MIN,IP_AMT_PAYMENT_MEAN_MAX,IP_AMT_PAYMENT_MEAN_MEAN,IP_AMT_PAYMENT_MEAN_SUM,IP_AMT_PAYMENT_SUM_MIN,IP_AMT_PAYMENT_SUM_MAX,IP_AMT_PAYMENT_SUM_MEAN,IP_AMT_PAYMENT_SUM_SUM
0,100018,4,1.000,0,0.000,0,0.00,0,0.0,0,...,49281.975,147845.925,4180.365000,81904.827857,44931.614286,134794.842857,20901.825,584515.800,392917.140,1178751.420
1,100019,1,1.000,0,0.000,0,0.00,0,0.0,0,...,44369.280,44369.280,20636.628750,20636.628750,20636.628750,20636.628750,165093.030,165093.030,165093.030,165093.030
2,100023,2,0.500,1,0.250,1,0.25,0,0.0,0,...,93183.285,279549.855,9046.992857,70571.711250,30064.665536,90193.996607,84602.340,282286.845,164515.695,493547.085
3,100027,2,0.500,0,0.000,2,0.50,0,0.0,0,...,24823.395,49646.790,21567.930000,28060.920000,24814.425000,49628.850000,258815.160,280609.200,269712.180,539424.360
4,100036,7,0.875,1,0.125,0,0.00,0,0.0,0,...,6551.415,6551.415,6551.133750,6551.133750,6551.133750,6551.133750,78613.605,78613.605,78613.605,78613.605


In [26]:
merge_dataset_with_previous_application_agg.columns.to_series().to_csv(
    'columns.csv', index=False, header=['column_name']
)

In [28]:
# merge application_train_transformed với merge_dataset_with_previous_application_agg và merge_dataset_with_bureau_agg
final_dataset = application_train_transformed.merge(merge_dataset_with_previous_application_agg, on='SK_ID_CURR', how='left')
final_dataset = final_dataset.merge(merge_dataset_with_bureau_agg, on='SK_ID_CURR', how='left')
final_dataset.head()

,AT_FLAG_PHONE,AT_DAYS_LAST_PHONE_CHANGE,AT_DEF_30_CNT_SOCIAL_CIRCLE,AT_AMT_CREDIT,AT_FLAG_WORK_PHONE,AT_OBS_30_CNT_SOCIAL_CIRCLE,AT_NAME_TYPE_SUITE,AT_DAYS_REGISTRATION,AT_CNT_CHILDREN,AT_NAME_INCOME_TYPE,...,BB_STATUS_C_MEAN_MEAN,BB_STATUS_C_MEAN_SUM,BB_STATUS_X_SUM_MIN,BB_STATUS_X_SUM_MAX,BB_STATUS_X_SUM_MEAN,BB_STATUS_X_SUM_SUM,BB_STATUS_X_MEAN_MIN,BB_STATUS_X_MEAN_MAX,BB_STATUS_X_MEAN_MEAN,BB_STATUS_X_MEAN_SUM
0,0,-1740.0,0.0,264888.0,0,2.0,Unaccompanied,-2684.0,1,Commercial associate,...,0.651700,11.078897,0.0,93.0,12.352941,210.0,0.000000,1.000000,0.170317,2.895382
1,1,-369.0,0.0,679500.0,0,3.0,Unaccompanied,-9211.0,0,Pensioner,...,0.145482,1.163856,3.0,18.0,10.875000,87.0,0.149425,1.000000,0.532039,4.256309
2,1,-158.0,0.0,239850.0,0,0.0,Unaccompanied,-2278.0,0,Pensioner,...,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
3,0,-540.0,0.0,454500.0,1,2.0,Unaccompanied,-8244.0,0,State servant,...,NaN,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.000000
4,0,-1562.0,0.0,590337.0,0,0.0,Family,-2828.0,0,State servant,...,0.000000,0.000000,2.0,9.0,5.500000,11.0,0.222222,0.529412,0.375817,0.751634
